# BetterCallNLI MS3 — Shard 4/5

This notebook processes **shard 3 of 5** of the ContractNLI test split. It downloads the latest bundled script from GitHub and writes results to `/kaggle/working/outputs/ms3_shard_3/`.

**Setup:**
1. Settings → Accelerator → **GPU T4 x2**
2. Add-ons → Secrets:
   - `CHROMA_API_KEY` (required)
   - `HF_TOKEN` (optional)
3. Run All cells
4. Output tab → download `ms3_shard_3.zip` → send to the merger.

Spec compliance (§2f): loads BASE `Qwen2.5-7B-Instruct` (no LoRA adapter).

In [ ]:
!pip install -q --upgrade unsloth unsloth_zoo
!pip install -q -U 'bitsandbytes>=0.46.1'
!pip install -q rich tqdm pandas pyyaml sentence-transformers chromadb neo4j huggingface_hub kagglehub

In [ ]:
import os
from pathlib import Path

SHARD_INDEX = 3
SHARD_TOTAL = 5
BUNDLE_PATH = Path('/kaggle/working/kaggle_ms3_eval.py')
OUTPUT_DIR  = Path(f'/kaggle/working/outputs/ms3_shard_{SHARD_INDEX}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for k in ('HF_TOKEN', 'CHROMA_API_KEY', 'NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD'):
        try: os.environ[k] = secrets.get_secret(k)
        except Exception: pass
    print('secrets loaded:', [k for k in ('HF_TOKEN','CHROMA_API_KEY','NEO4J_URI') if os.getenv(k)])
except ImportError:
    print('kaggle_secrets unavailable; assuming env vars are already set')
os.environ.setdefault('HF_TOKEN', 'local-model-stub')

!wget -q 'https://raw.githubusercontent.com/MohamedAbdel-Azeem/BetterCallNLI/feat/cli-evaluation/kaggle_ms3_eval.py' -O /kaggle/working/kaggle_ms3_eval.py
print('bundle size :', BUNDLE_PATH.stat().st_size, 'bytes')
print(f'shard       : {SHARD_INDEX + 1}/{SHARD_TOTAL}')
print(f'output dir  : {OUTPUT_DIR}')

In [ ]:
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('unsloth').setLevel(logging.ERROR)

# Checkpointing is on by default — re-running this cell after a kernel timeout
# picks up from where it stopped (see <output_dir>/checkpoint.json).
!python /kaggle/working/kaggle_ms3_eval.py --retrieval vector --shard-index 3 --shard-total 5 --output-dir /kaggle/working/outputs/ms3_shard_3 --max-seq-len 8192

In [ ]:
import shutil
from pathlib import Path

shard_dir = Path('/kaggle/working/outputs/ms3_shard_3')
zip_path = shutil.make_archive('/kaggle/working/ms3_shard_3', 'zip', root_dir=shard_dir)
print('Send this file to the merger:', zip_path)
print('Files inside:')
for p in sorted(shard_dir.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(shard_dir)}  ({p.stat().st_size:,} bytes)')

## When done

1. Open **Output** panel → download `ms3_shard_3.zip`.
2. Send it to whoever's running the merge step.

The merger extracts all 5 zips into one directory and runs:
```bash
python scripts/merge_shards.py \n    --shards-parent results/ms3/shards \n    --output-dir results/ms3/merged
```
to produce the §5b combined CSV and §5c runtraces zip.